# NB01 · Baseline léxico

**Evidencia para D05** (qué baselines léxicos se implementan). Aquí no se implementa ningún retriever todavía: solo se mide qué tienen los datos que separa unas opciones de otras. La implementación llega cuando D05 esté ratificada.

---

### 📐 Convención de corpus — qué se mide sobre qué

Toda cabecera de sección lleva marcado su corpus, y ninguna celda mezcla los dos:

| Marca | Corpus | Fichero | Para qué |
|---|---|---|---|
| 🔬 **MUESTRA** | 1.500 registros | `catalogo_muestra.csv` | Desarrollo y calibración (condición 3 del plan) |
| 📚 **COMPLETO** | 15.000 registros | `catalogo_productos.csv` | Confirmación de las decisiones antes de fijarlas |

La **Parte A** mide sobre la muestra y la **Parte B** repite exactamente las mismas pruebas sobre el catálogo completo.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import pandas as pd

from aurum.datos import (
    document_length_stats,
    literal_match_ceiling,
    query_term_coverage,
    query_token_frequencies,
)

DATA = Path("..") / "data"

# Los dos corpus, cargados con nombres que no se pueden confundir.
muestra = pd.read_csv(DATA / "catalogo_muestra.csv")
completo = pd.read_csv(DATA / "catalogo_productos.csv")

consultas = pd.read_csv(DATA / "consultas_desarrollo.csv")
relevancias = pd.read_csv(DATA / "relevancias_desarrollo.csv")
consultas_eval = pd.read_csv(DATA / "consultas_evaluacion.csv")

print(f"🔬 MUESTRA : {len(muestra):>6} registros")
print(f"📚 COMPLETO: {len(completo):>6} registros")
print(f"consultas_desarrollo: {len(consultas)} · consultas_evaluacion: {len(consultas_eval)}")


🔬 MUESTRA :   1500 fichas
📚 COMPLETO:  15000 fichas
consultas_desarrollo: 8 · consultas_evaluacion: 12


---

# Parte A · Pruebas sobre la 🔬 MUESTRA (1.500 registros)

## A.1 · 🔬 Dispersión de la longitud de documento

BM25 normaliza la longitud del documento de forma explícita (parámetro `b`); TF-IDF solo la absorbe en la norma L2 del vector. Cuanto mayor sea la dispersión (`cv`, `ratio_p95_p50`), más se separan ambos métodos y más sentido tiene implementar los dos en vez de uno.

In [2]:
document_length_stats(muestra, ["title", "text"])


,campo,n_docs,tokens_media,tokens_p50,tokens_p90,tokens_p95,tokens_max,cv,ratio_p95_p50
0,title,1500,19.7,20,31,33,48,0.444,1.65
1,text,1500,213.1,171,475,496,603,0.777,2.90


## A.2 · 🔬 Cobertura léxica de las 8 consultas de desarrollo

Un término que no está en el vocabulario del corpus no aporta señal ni a TF-IDF ni a BM25. Se mide con y sin `strip_accents` porque varias consultas vienen **sin tildes** (`habitacion`, `tacon`, `tactil`) mientras el catálogo sí las lleva: es una decisión de configuración del índice, no un detalle cosmético.

`df_min` es la frecuencia documental del término **más raro** de la consulta: el que más peso IDF recibe y, por tanto, el que más manda en el ranking.

In [3]:
cobertura_con_tildes = query_term_coverage(consultas, muestra)
cobertura_sin_tildes = query_term_coverage(consultas, muestra, strip_accents=True)

comparativa_muestra = cobertura_con_tildes[
    ["query_id", "query_text", "n_tokens", "n_oov", "tokens_oov", "df_min", "df_max"]
].merge(
    cobertura_sin_tildes[["query_id", "n_oov", "tokens_oov", "df_min"]],
    on="query_id",
    suffixes=("_con_tildes", "_sin_tildes"),
)
comparativa_muestra


,query_id,query_text,n_tokens,n_oov_con_tildes,tokens_oov_con_tildes,df_min_con_tildes,df_max,n_oov_sin_tildes,tokens_oov_sin_tildes,df_min_sin_tildes
0,13357,base tapizada 160x200 sin patas,5,1,160x200,29,548,1,160x200,29
1,18868,botines marrones mujer tacon medio,5,0,,4,132,0,,5
2,28703,convertibles 2 en 1 portátil tactil,6,1,convertibles,3,1018,1,convertibles,49
3,31224,cámaras bridge baratas,3,0,,4,20,0,,4
4,33633,disfraz halloween talla grande hombre,5,0,,17,95,0,,17
5,38249,estantes sin taladro habitacion,4,0,,2,548,0,,28
6,43240,funda ipad air 4 sin tapa,6,0,,52,548,0,,52
7,61533,lentejas sin gluten,3,0,,26,548,0,,26


### A.2b · 🔬 La misma cobertura, palabra por palabra

La vista desagregada: a qué lista de documentos apunta **cada palabra** que escribe el usuario. `df = 0` es una palabra que no existe en el corpus; un `df` muy pequeño recibe mucho peso IDF — para bien (término preciso) o para mal (errata). Comparar las dos columnas muestra qué palabras cambian de lista al normalizar acentos.

In [4]:
tokens_dev_muestra = query_token_frequencies(consultas, muestra)
tokens_dev_muestra["cambia_al_normalizar"] = (
    tokens_dev_muestra["df_con_tildes"] != tokens_dev_muestra["df_sin_tildes"]
)
tokens_dev_muestra.merge(consultas[["query_id", "query_text"]], on="query_id")[
    ["query_id", "query_text", "posicion", "token", "df_con_tildes", "df_sin_tildes", "cambia_al_normalizar"]
]


,query_id,query_text,posicion,token,df_con_tildes,df_sin_tildes,cambia_al_normalizar
0,13357,base tapizada 160x200 sin patas,1,base,110,110,False
1,13357,base tapizada 160x200 sin patas,2,tapizada,29,29,False
2,13357,base tapizada 160x200 sin patas,3,160x200,0,0,False
3,13357,base tapizada 160x200 sin patas,4,sin,548,548,False
4,13357,base tapizada 160x200 sin patas,5,patas,65,65,False
5,18868,botines marrones mujer tacon medio,1,botines,12,12,False
6,18868,botines marrones mujer tacon medio,2,marrones,5,5,False
7,18868,botines marrones mujer tacon medio,3,mujer,132,132,False
8,18868,botines marrones mujer tacon medio,4,tacon,4,24,True
9,18868,botines marrones mujer tacon medio,5,medio,80,80,False


## A.3 · 🔬 Cobertura de las 12 consultas ciegas (`direct` · `context` · `semantic`)

Sin etiquetas no se puede calcular nDCG sobre ellas, pero la cobertura léxica sí se mide: es la evidencia directa del *vocabulary gap* que justifica el coste del sistema denso.

In [5]:
cobertura_eval_muestra = query_term_coverage(
    consultas_eval, muestra, id_col="evaluation_id", strip_accents=True
)
cobertura_eval_muestra["formulacion"] = (
    cobertura_eval_muestra["evaluation_id"].str.split("-").str[2]
)
cobertura_eval_muestra[
    ["evaluation_id", "formulacion", "query_text", "n_tokens", "n_oov", "tokens_oov", "df_min", "df_max"]
].sort_values("evaluation_id")


,evaluation_id,formulacion,query_text,n_tokens,n_oov,tokens_oov,df_min,df_max
0,EVAL-100455-context,context,taladro sin cable de 24 voltios que venga con ...,11,0,,1,1341
1,EVAL-100455-direct,direct,taladro 24v batería,3,0,,7,99
2,EVAL-100455-semantic,semantic,quiero una herramienta inalámbrica potente par...,12,2,"quiero, depender",16,1341
3,EVAL-101352-context,context,"tele de tamaño reducido para una cocina, alred...",11,1,tele,4,1341
4,EVAL-101352-direct,direct,television 28 pulgadas,3,0,,10,161
5,EVAL-101352-semantic,semantic,busco un televisor pequeño de unas setenta cen...,11,2,"busco, setenta",2,1341
6,EVAL-93437-context,context,me duele la espalda al trabajar y necesito una...,14,2,"duele, necesito",3,1152
7,EVAL-93437-direct,direct,sillas oficina ergonomicas,3,0,,3,99
8,EVAL-93437-semantic,semantic,necesito un asiento cómodo para trabajar ocho ...,14,1,necesito,2,1131
9,EVAL-96202-context,context,pieza para sujetar un aire acondicionado en el...,12,0,,5,1341


In [6]:
cobertura_eval_muestra.groupby("formulacion")[
    ["n_tokens", "n_oov", "df_min", "df_max"]
].mean().round(1)


,n_tokens,n_oov,df_min,df_max
formulacion,,,,
context,12.0,0.8,3.2,1293.8
direct,3.2,0.0,15.5,130.5
semantic,13.0,1.2,5.5,1288.5


### A.3b · 🔬 Las palabras de las ciegas que no existen en el corpus

Qué escribe el cliente que no está en ningún registro del catálogo. Es el *vocabulary gap* en su forma más literal.

In [7]:
tokens_eval_muestra = query_token_frequencies(
    consultas_eval, muestra, id_col="evaluation_id"
)
tokens_eval_muestra["formulacion"] = (
    tokens_eval_muestra["evaluation_id"].str.split("-").str[2]
)
tokens_eval_muestra[tokens_eval_muestra["df_sin_tildes"] == 0][
    ["evaluation_id", "formulacion", "posicion", "token"]
]


,evaluation_id,formulacion,posicion,token
14,EVAL-100455-semantic,semantic,1,quiero
22,EVAL-100455-semantic,semantic,9,depender
26,EVAL-101352-context,context,1,tele
40,EVAL-101352-semantic,semantic,1,busco
46,EVAL-101352-semantic,semantic,7,setenta
52,EVAL-93437-context,context,2,duele
58,EVAL-93437-context,context,8,necesito
68,EVAL-93437-semantic,semantic,1,necesito


## A.4 · 🔬 Techo del emparejamiento literal

% de productos **relevantes** (E+S, según D01) que contienen *todos* los términos de la consulta. Es el techo de la opción *«coincidencia exacta de título»* y la medida de cuánto queda fuera del alcance de una coincidencia literal.

In [8]:
techo_muestra = literal_match_ceiling(consultas, relevancias, muestra, strip_accents=True)
techo_muestra[
    ["query_id", "query_text", "n_relevantes", "n_en_catalogo",
     "n_todos_en_title", "pct_todos_en_title", "n_todos_en_text", "pct_todos_en_text"]
]


,query_id,query_text,n_relevantes,n_en_catalogo,n_todos_en_title,pct_todos_en_title,n_todos_en_text,pct_todos_en_text
0,13357,base tapizada 160x200 sin patas,31,31,0,0.0,0,0.0
1,18868,botines marrones mujer tacon medio,9,9,0,0.0,0,0.0
2,28703,convertibles 2 en 1 portátil tactil,39,39,0,0.0,0,0.0
3,31224,cámaras bridge baratas,15,15,0,0.0,0,0.0
4,33633,disfraz halloween talla grande hombre,4,4,0,0.0,0,0.0
5,38249,estantes sin taladro habitacion,35,35,0,0.0,1,2.9
6,43240,funda ipad air 4 sin tapa,35,35,0,0.0,13,37.1
7,61533,lentejas sin gluten,30,30,11,36.7,16,53.3


---

# Parte B · Las mismas pruebas sobre el 📚 CATÁLOGO COMPLETO (15.000 registros)

Mismo código, mismas consultas, único factor que cambia: el corpus. Sirve para comprobar si las conclusiones de la Parte A se sostienen a escala real o eran un artefacto del tamaño de la muestra — el mismo procedimiento que se siguió con D03 en NB00.

⏱️ Estas celdas tardan más: recorren 15.000 registros en vez de 1.500.

## B.1 · 📚 Dispersión de la longitud de documento

In [9]:
document_length_stats(completo, ["title", "text"])


,campo,n_docs,tokens_media,tokens_p50,tokens_p90,tokens_p95,tokens_max,cv,ratio_p95_p50
0,title,15000,18.5,18,30,32,80,0.465,1.78
1,text,15000,196.7,150,466,493,603,0.828,3.29


## B.2 · 📚 Cobertura léxica de las 8 consultas de desarrollo

In [10]:
cobertura_con_tildes_completo = query_term_coverage(consultas, completo)
cobertura_sin_tildes_completo = query_term_coverage(consultas, completo, strip_accents=True)

comparativa_completo = cobertura_con_tildes_completo[
    ["query_id", "query_text", "n_tokens", "n_oov", "tokens_oov", "df_min", "df_max"]
].merge(
    cobertura_sin_tildes_completo[["query_id", "n_oov", "tokens_oov", "df_min"]],
    on="query_id",
    suffixes=("_con_tildes", "_sin_tildes"),
)
comparativa_completo


,query_id,query_text,n_tokens,n_oov_con_tildes,tokens_oov_con_tildes,df_min_con_tildes,df_max,n_oov_sin_tildes,tokens_oov_sin_tildes,df_min_sin_tildes
0,13357,base tapizada 160x200 sin patas,5,0,,2,4583,0,,2
1,18868,botines marrones mujer tacon medio,5,0,,16,1296,0,,25
2,28703,convertibles 2 en 1 portátil tactil,6,0,,2,9644,0,,2
3,31224,cámaras bridge baratas,3,0,,14,180,0,,14
4,33633,disfraz halloween talla grande hombre,5,0,,174,1118,0,,174
5,38249,estantes sin taladro habitacion,4,0,,19,4583,0,,109
6,43240,funda ipad air 4 sin tapa,6,0,,186,4583,0,,186
7,61533,lentejas sin gluten,3,0,,27,4583,0,,27


### B.2b · 📚 La misma cobertura, palabra por palabra

In [11]:
tokens_dev_completo = query_token_frequencies(consultas, completo)
tokens_dev_completo["cambia_al_normalizar"] = (
    tokens_dev_completo["df_con_tildes"] != tokens_dev_completo["df_sin_tildes"]
)
tokens_dev_completo.merge(consultas[["query_id", "query_text"]], on="query_id")[
    ["query_id", "query_text", "posicion", "token", "df_con_tildes", "df_sin_tildes", "cambia_al_normalizar"]
]


,query_id,query_text,posicion,token,df_con_tildes,df_sin_tildes,cambia_al_normalizar
0,13357,base tapizada 160x200 sin patas,1,base,818,818,False
1,13357,base tapizada 160x200 sin patas,2,tapizada,40,40,False
2,13357,base tapizada 160x200 sin patas,3,160x200,2,2,False
3,13357,base tapizada 160x200 sin patas,4,sin,4583,4583,False
4,13357,base tapizada 160x200 sin patas,5,patas,235,235,False
5,18868,botines marrones mujer tacon medio,1,botines,46,46,False
6,18868,botines marrones mujer tacon medio,2,marrones,25,25,False
7,18868,botines marrones mujer tacon medio,3,mujer,1296,1300,True
8,18868,botines marrones mujer tacon medio,4,tacon,16,140,True
9,18868,botines marrones mujer tacon medio,5,medio,738,738,False


## B.3 · 📚 Cobertura de las 12 consultas ciegas

In [12]:
cobertura_eval_completo = query_term_coverage(
    consultas_eval, completo, id_col="evaluation_id", strip_accents=True
)
cobertura_eval_completo["formulacion"] = (
    cobertura_eval_completo["evaluation_id"].str.split("-").str[2]
)
cobertura_eval_completo[
    ["evaluation_id", "formulacion", "query_text", "n_tokens", "n_oov", "tokens_oov", "df_min", "df_max"]
].sort_values("evaluation_id")


,evaluation_id,formulacion,query_text,n_tokens,n_oov,tokens_oov,df_min,df_max
0,EVAL-100455-context,context,taladro sin cable de 24 voltios que venga con ...,11,0,,11,13144
1,EVAL-100455-direct,direct,taladro 24v batería,3,0,,44,874
2,EVAL-100455-semantic,semantic,quiero una herramienta inalámbrica potente par...,12,0,,2,13144
3,EVAL-101352-context,context,"tele de tamaño reducido para una cocina, alred...",11,0,,6,13144
4,EVAL-101352-direct,direct,television 28 pulgadas,3,0,,98,1213
5,EVAL-101352-semantic,semantic,busco un televisor pequeño de unas setenta cen...,11,1,busco,1,13144
6,EVAL-93437-context,context,me duele la espalda al trabajar y necesito una...,14,0,,1,11203
7,EVAL-93437-direct,direct,sillas oficina ergonomicas,3,0,,22,763
8,EVAL-93437-semantic,semantic,necesito un asiento cómodo para trabajar ocho ...,14,0,,12,11182
9,EVAL-96202-context,context,pieza para sujetar un aire acondicionado en el...,12,0,,64,13144


In [13]:
cobertura_eval_completo.groupby("formulacion")[
    ["n_tokens", "n_oov", "df_min", "df_max"]
].mean().round(1)


,n_tokens,n_oov,df_min,df_max
formulacion,,,,
context,12.0,0.0,20.5,12658.8
direct,3.2,0.0,62.0,1018.0
semantic,13.0,0.2,9.0,12653.5


### B.3b · 📚 Las palabras de las ciegas que no existen en el corpus

In [14]:
tokens_eval_completo = query_token_frequencies(
    consultas_eval, completo, id_col="evaluation_id"
)
tokens_eval_completo["formulacion"] = (
    tokens_eval_completo["evaluation_id"].str.split("-").str[2]
)
tokens_eval_completo[tokens_eval_completo["df_sin_tildes"] == 0][
    ["evaluation_id", "formulacion", "posicion", "token"]
]


,evaluation_id,formulacion,posicion,token
40,EVAL-101352-semantic,semantic,1,busco


## B.4 · 📚 Techo del emparejamiento literal

In [15]:
techo_completo = literal_match_ceiling(consultas, relevancias, completo, strip_accents=True)
techo_completo[
    ["query_id", "query_text", "n_relevantes", "n_en_catalogo",
     "n_todos_en_title", "pct_todos_en_title", "n_todos_en_text", "pct_todos_en_text"]
]


,query_id,query_text,n_relevantes,n_en_catalogo,n_todos_en_title,pct_todos_en_title,n_todos_en_text,pct_todos_en_text
0,13357,base tapizada 160x200 sin patas,31,31,0,0.0,0,0.0
1,18868,botines marrones mujer tacon medio,9,9,0,0.0,0,0.0
2,28703,convertibles 2 en 1 portátil tactil,39,39,0,0.0,0,0.0
3,31224,cámaras bridge baratas,15,15,0,0.0,0,0.0
4,33633,disfraz halloween talla grande hombre,4,4,0,0.0,0,0.0
5,38249,estantes sin taladro habitacion,35,35,0,0.0,1,2.9
6,43240,funda ipad air 4 sin tapa,35,35,0,0.0,13,37.1
7,61533,lentejas sin gluten,30,30,11,36.7,16,53.3


---

# Parte C · El baseline léxico

Decisiones ratificadas y escritas en `config/config.yaml`:

| ID | Decisión | Evidencia que la sostiene |
|---|---|---|
| **D05** | **TF-IDF + BM25** | Son los dos únicos que se diferencian en algo medible aquí: la normalización de longitud, con 3,29× de dispersión en `text` (B.1). LSA heredaría el vocabulario y la normalización de TF-IDF; la coincidencia exacta daría 0 en 7 de 8 consultas (B.4) |
| **D05.b** | **Normalizar acentos** al indexar | `tactil` pasa de 10 a 326 registros, `habitacion` de 19 a 434 (B.2b). Las palabras ya bien escritas apenas se mueven |
| **D05.c** | Índice sobre **`text`** | Es la misma superficie textual que usará el denso en NB02 (plantilla A0), así que la comparación léxico↔denso varía un solo factor |

Los dos retrievers comparten tokenizador (`aurum.datos.tokenize`), así que ven exactamente los mismos términos: lo único que cambia entre ellos es la fórmula de puntuación (Regla 2 de experimentación).

**Contrato de relevancia** (`manifest.json`): `E=3, S=2, C=1, I=0`. **D01**: relevante para Recall@10/MRR@10 es `E+S`. **D04**: un producto recuperado sin juicio puntúa 0.

In [16]:
from aurum.evaluacion import (
    evaluate_rankings,
    formulation_consistency,
    qrels_from_judgements,
)
from aurum.lexico import Bm25Retriever, TfidfRetriever, rank_queries

CAMPO = "text"   # D05.c
STRIP_ACCENTS = True  # D05.b
TOP_K = 10

qrels = qrels_from_judgements(relevancias)
print(f"qrels: {len(qrels)} consultas juzgadas, "
      f"{sum(len(v) for v in qrels.values())} juicios")


qrels: 8 consultas juzgadas, 248 juicios


## C.1 · 🔬 Construcción de los dos índices sobre la muestra

Se cronometra la construcción: el coste de indexación es parte de la comparación, no una nota al pie.

In [17]:
import time

def construir(clase, catalogo):
    inicio = time.perf_counter()
    retriever = clase(
        catalogo[CAMPO].tolist(),
        catalogo["product_id"].tolist(),
        strip_accents=STRIP_ACCENTS,
    )
    return retriever, time.perf_counter() - inicio

tfidf_muestra, t_tfidf = construir(TfidfRetriever, muestra)
bm25_muestra, t_bm25 = construir(Bm25Retriever, muestra)

pd.DataFrame([
    {"baseline": "tfidf", "n_docs": len(muestra), "vocabulario": tfidf_muestra.vocabulary_size,
     "construccion_s": round(t_tfidf, 2)},
    {"baseline": "bm25", "n_docs": len(muestra), "vocabulario": bm25_muestra.vocabulary_size,
     "construccion_s": round(t_bm25, 2)},
])


,baseline,n_docs,vocabulario,construccion_s
0,tfidf,1500,22978,0.71
1,bm25,1500,22978,0.64


## C.2 · 🔬 Métricas sobre las 8 consultas de desarrollo

La media macro primero, y **la tabla por consulta justo después**: la consulta 33633 tiene un solo `Exact`, así que su Recall@10 solo puede valer 0 o 1 y distorsiona la media (trampa nº 8 del plan).

In [18]:
rankings_muestra = {
    "tfidf": rank_queries(tfidf_muestra, consultas, k=TOP_K),
    "bm25": rank_queries(bm25_muestra, consultas, k=TOP_K),
}
informes_muestra = {
    nombre: evaluate_rankings(ranking, qrels, k=TOP_K)
    for nombre, ranking in rankings_muestra.items()
}

pd.DataFrame([
    {"baseline": nombre, **informe.summary}
    for nombre, informe in informes_muestra.items()
])


,baseline,precision_at_10,recall_at_10,mrr_at_10,ndcg_at_10
0,tfidf,0.6000,0.2324,0.8750,0.5654
1,bm25,0.7125,0.3130,0.8906,0.6512


### C.2b · 🔬 Tabla por consulta

In [19]:
por_consulta = pd.concat([
    informe.per_query_frame().assign(baseline=nombre)
    for nombre, informe in informes_muestra.items()
])
por_consulta["query_id"] = por_consulta["query_id"].astype(int)
por_consulta.merge(consultas[["query_id", "query_text"]], on="query_id").pivot(
    index=["query_id", "query_text"], columns="baseline",
    values=["ndcg@10", "recall@10", "mrr@10"],
)


ndcg@10         recall@10  \
baseline                                          bm25   tfidf      bm25   
query_id query_text                                                        
13357    base tapizada 160x200 sin patas        0.5441  0.5006    0.2581   
18868    botines marrones mujer tacon medio     0.5270  0.3634    0.5556   
28703    convertibles 2 en 1 portátil tactil    0.8709  0.7975    0.2308   
31224    cámaras bridge baratas                 0.5777  0.5712    0.3333   
33633    disfraz halloween talla grande hombre  0.1566  0.0254    0.2500   
38249    estantes sin taladro habitacion        0.8125  0.5343    0.2571   
43240    funda ipad air 4 sin tapa              0.7208  0.7311    0.2857   
61533    lentejas sin gluten                    1.0000  1.0000    0.3333   

                                                       mrr@10        
baseline                                         tfidf   bm25 tfidf  
query_id query_text                                                  
13357    base tapizada 160x200 sin patas        0.2258  1.000   1.0  
18868    botines marrones mujer tacon medio     0.3333  1.000   1.0  
28703    convertibles 2 en 1 portátil tactil    0.2051  1.000   1.0  
31224    cámaras bridge baratas                 0.3333  1.000   1.0  
33633    disfraz halloween talla grande hombre  0.0000  0.125   0.0  
38249    estantes sin taladro habitacion        0.1714  1.000   1.0  
43240    funda ipad air 4 sin tapa              0.2571  1.000   1.0  
61533    lentejas sin gluten                    0.3333  1.000   1.0

## C.3 · 📚 Las mismas métricas sobre el catálogo completo

Aquí el buscador compite contra 15.000 candidatos en vez de 1.500: es el escenario real y las métricas deberían bajar. La diferencia entre C.2 y C.3 mide cuánto de la calidad venía de que el corpus fuera pequeño.

In [20]:
tfidf_completo, t_tfidf_c = construir(TfidfRetriever, completo)
bm25_completo, t_bm25_c = construir(Bm25Retriever, completo)

rankings_completo = {
    "tfidf": rank_queries(tfidf_completo, consultas, k=TOP_K),
    "bm25": rank_queries(bm25_completo, consultas, k=TOP_K),
}
informes_completo = {
    nombre: evaluate_rankings(ranking, qrels, k=TOP_K)
    for nombre, ranking in rankings_completo.items()
}

pd.DataFrame([
    {"baseline": nombre, "n_docs": len(completo), **informe.summary}
    for nombre, informe in informes_completo.items()
])


,baseline,n_docs,precision_at_10,recall_at_10,mrr_at_10,ndcg_at_10
0,tfidf,15000,0.4125,0.1510,0.75,0.4129
1,bm25,15000,0.5500,0.1841,0.75,0.5088


### C.3b · 📚 Tabla por consulta

In [21]:
por_consulta_completo = pd.concat([
    informe.per_query_frame().assign(baseline=nombre)
    for nombre, informe in informes_completo.items()
])
por_consulta_completo["query_id"] = por_consulta_completo["query_id"].astype(int)
por_consulta_completo.merge(consultas[["query_id", "query_text"]], on="query_id").pivot(
    index=["query_id", "query_text"], columns="baseline",
    values=["ndcg@10", "recall@10", "mrr@10"],
)


ndcg@10         recall@10  \
baseline                                          bm25   tfidf      bm25   
query_id query_text                                                        
13357    base tapizada 160x200 sin patas        0.5490  0.5040    0.2581   
18868    botines marrones mujer tacon medio     0.1617  0.2929    0.1111   
28703    convertibles 2 en 1 portátil tactil    0.7237  0.4986    0.1795   
31224    cámaras bridge baratas                 0.2566  0.1737    0.1333   
33633    disfraz halloween talla grande hombre  0.0233  0.0000    0.0000   
38249    estantes sin taladro habitacion        0.6865  0.4097    0.2000   
43240    funda ipad air 4 sin tapa              0.6693  0.4243    0.2571   
61533    lentejas sin gluten                    1.0000  1.0000    0.3333   

                                                       mrr@10        
baseline                                         tfidf   bm25 tfidf  
query_id query_text                                                  
13357    base tapizada 160x200 sin patas        0.2258    1.0   1.0  
18868    botines marrones mujer tacon medio     0.2222    0.5   1.0  
28703    convertibles 2 en 1 portátil tactil    0.1026    1.0   1.0  
31224    cámaras bridge baratas                 0.0667    0.5   0.5  
33633    disfraz halloween talla grande hombre  0.0000    0.0   0.0  
38249    estantes sin taladro habitacion        0.1429    1.0   0.5  
43240    funda ipad air 4 sin tapa              0.1143    1.0   1.0  
61533    lentejas sin gluten                    0.3333    1.0   1.0

## C.4 · 📚 Coherencia entre formulaciones de las 12 ciegas (Jaccard@10)

Las 12 consultas de evaluación **no tienen juicios**, así que su nDCG es incalculable. Lo que sí se mide sin etiquetas: las tres formulaciones de una misma intención piden lo mismo, así que un buscador que entienda la intención debería devolver productos parecidos. Jaccard@10 = productos en ambos top-10 / productos en alguno de los dos.

In [22]:
consistencia = {
    nombre: formulation_consistency(
        rank_queries(retriever, consultas_eval, id_col="evaluation_id", k=TOP_K),
        k=TOP_K,
    ).assign(baseline=nombre)
    for nombre, retriever in [("tfidf", tfidf_completo), ("bm25", bm25_completo)]
}
pd.concat(consistencia.values()).set_index(["baseline", "intencion"]).sort_index()


jaccard_context_direct  jaccard_context_semantic  \
baseline intencion                                                     
bm25     100455                     0.2500                    0.0000   
         101352                     0.0526                    0.0000   
         93437                      0.0000                    0.2500   
         96202                      0.3333                    0.6667   
tfidf    100455                     0.8182                    0.0526   
         101352                     0.0526                    0.0000   
         93437                      0.0000                    0.2500   
         96202                      0.6667                    0.5385   

                    jaccard_direct_semantic  
baseline intencion                           
bm25     100455                      0.0000  
         101352                      0.0000  
         93437                       0.0000  
         96202                       0.3333  
tfidf    100455                      0.0526  
         101352                      0.0000  
         93437                       0.0000  
         96202                       0.5385

## C.5 · Artefacto `artifacts/baseline_lexico.json`

Métricas **y los IDs recuperados por consulta**: sin los IDs no se puede atribuir errores en NB09 (Regla 3 de experimentación).

In [23]:
import json

artefacto = {
    "configuracion": {
        "campo_indexado": CAMPO,
        "strip_accents": STRIP_ACCENTS,
        "top_k": TOP_K,
        "relevancia": {"E": 3, "S": 2, "C": 1, "I": 0},
        "umbral_relevante_recall_mrr": 2.0,
        "gain_ndcg": "exponential",
    },
    "muestra": {
        "n_docs": len(muestra),
        "metricas": {n: i.summary for n, i in informes_muestra.items()},
        "por_consulta": {n: i.per_query_frame().to_dict("records") for n, i in informes_muestra.items()},
        "rankings": rankings_muestra,
    },
    "completo": {
        "n_docs": len(completo),
        "metricas": {n: i.summary for n, i in informes_completo.items()},
        "por_consulta": {n: i.per_query_frame().to_dict("records") for n, i in informes_completo.items()},
        "rankings": rankings_completo,
        "jaccard_ciegas": pd.concat(consistencia.values()).to_dict("records"),
    },
}

destino = Path("..") / "artifacts" / "baseline_lexico.json"
destino.write_text(json.dumps(artefacto, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Escrito {destino} ({destino.stat().st_size / 1024:.1f} KB)")


Escrito ..\artifacts\baseline_lexico.json (17.1 KB)
